In [1]:
import socket
import threading
import time

In [2]:
HOST = "127.0.0.1"
PORT = 65001

In [3]:
max_worker = 2

In [4]:
worker_slot = threading.Semaphore(max_worker)

In [5]:
def handle_client(conn,addr):
    with conn:
        request = conn.recv(1024).decode()
        print(f"Worker-[{threading.get_ident()}] waiting for a free slot {addr}")
        
        got_slot = worker_slot.acquire(timeout=3)

        if not got_slot:
            print(f"[Worker-{threading.get_ident()}] no free slot, rejecting {addr}")
            conn.sendall(b"Server busy: Please try later")
            return
        try:
            print(f"[Worker-{threading.get_ident()}] got a  slot,working on {addr}]")
            time.sleep(2)
            reply = f"processed {request} by worker thread {threading.get_ident()}"
            conn.sendall(reply.encode())
            print(f"[Worker-{threading.get_ident()}] done with {addr},releasing slot")
        finally:
            worker_slot.release()

In [6]:
def main():
    with socket.socket(socket.AF_INET,socket.SOCK_STREAM) as server_socket:
        server_socket.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
        server_socket.bind((HOST,PORT))
        server_socket.listen(5)
        print(f"[Dispatcher] is listening on {HOST}:{PORT} max {max_worker} concurrent workers")

        while True:
            conn,addr = server_socket.accept()
            print(f"[Dispatcher] accepted address {addr},spawning worker thread")
            worker = threading.Thread(target = handle_client, args = (conn,addr))
            worker.start()

In [ ]:
if __name__ == "__main__":
    main()

[Dispatcher] is listening on 127.0.0.1:65001 max 2 concurrent workers
[Dispatcher] accepted address ('127.0.0.1', 6759),spawning worker thread
Worker-[8580] waiting for a free slot ('127.0.0.1', 6759)
[Worker-8580] got a  slot,working on ('127.0.0.1', 6759)]
[Dispatcher] accepted address ('127.0.0.1', 6760),spawning worker thread
Worker-[1548] waiting for a free slot ('127.0.0.1', 6760)
[Worker-1548] got a  slot,working on ('127.0.0.1', 6760)]
[Dispatcher] accepted address ('127.0.0.1', 6761),spawning worker thread
Worker-[4440] waiting for a free slot ('127.0.0.1', 6761)
[Dispatcher] accepted address ('127.0.0.1', 6762),spawning worker thread
Worker-[2116] waiting for a free slot ('127.0.0.1', 6762)
[Dispatcher] accepted address ('127.0.0.1', 6763),spawning worker thread
Worker-[6768] waiting for a free slot ('127.0.0.1', 6763)
[Worker-8580] done with ('127.0.0.1', 6759),releasing slot
[Worker-4440] got a  slot,working on ('127.0.0.1', 6761)]
[Worker-1548] done with ('127.0.0.1', 6760